# TN1 — CNN-LSTM ở ngân sách nhỏ

## Câu hỏi

TN1 cho thấy cơ chế hồi quy hợp bài toán này hơn tích chập thuần:

| cấu hình | tham số | cv_score | GHIJ |
|---|---|---|---|
| LSTM-352 | 1.502.713 | 0.7570 ± 0.0041 | 0.8103 ± 0.0154 |
| LSTM-67 | 56.908 | 0.7532 ± 0.0020 | 0.8017 ± 0.0025 |
| DS-TCN-64 | 56.281 | 0.7421 ± 0.0007 | 0.7958 ± 0.0154 |
| TCN-64 | 151.513 | 0.7423 ± 0.0044 | chưa chạy |

Câu hỏi tiếp:

> **Ghép tích chập làm bộ rút đặc trưng cho LSTM thì sao?**

Không thay hồi quy — giữ nó làm xương sống, dùng tích chập cho nó một đầu vào
tốt hơn.

## Tên gọi

Đây **không phải** `ConvLSTM`. `ConvLSTM` (Shi et al. 2015, NeurIPS) đưa tích
chập vào **bên trong** ô LSTM, dùng cho dữ liệu không-thời gian. Ở đây tích chập
đứng **trước**, LSTM vẫn là LSTM thường.

Tên đúng là **CNN-LSTM**. Tiền lệ ghép tích chập với hồi quy: **CLDNN** —
Sainath, Vinyals, Senior, Sak (2015), ICASSP, *"Convolutional, Long Short-Term
Memory, Fully Connected Deep Neural Networks"*. Khác CLDNN ở chỗ bỏ khối DNN
phía sau, thay bằng một tầng tuyến tính, vì bài toán chỉ cần xuất 25 mẫu.

## Kiến trúc

```
(batch, 200)
  -> Conv1d(1,  32, kernel=5, stride=2, padding=2) -> BatchNorm -> ReLU
  -> Conv1d(32, 32, kernel=5, stride=2, padding=2) -> BatchNorm -> ReLU
                                                          (batch, 32, 50)
  -> đổi trục                                             (batch, 50, 32)
  -> LSTM(input_size=32, hidden=58, 2 tầng, MỘT chiều)
  -> output[:, -1, :]                                     (batch, 58)
  -> Linear(58, 25)                                       (batch, 25)
```

`output[:, -1, :]` ở đây **là đúng** vì LSTM một chiều — bước cuối đã đọc hết
chuỗi. Chỗ phải tránh cách lấy này là LSTM hai chiều, xem `TN1_BiLSTM.ipynb`.

Hai tầng đều `stride=2` nên 200 xuống 100 xuống 50. **Không có pooling thêm.**

## Lý do từng con số

| | | |
|---|---|---|
| 2 tầng, stride 2 | 200 xuống 50 | một nhịp thở (200 mẫu ở 50 Hz) còn khoảng 12 bước, không nén quá tay |
| kernel 5 | phủ 0,1 giây | cỡ một đoạn dốc của sóng thở |
| 32 kênh | phần tích chập nhẹ | dồn ngân sách cho LSTM, thứ TN1 chứng minh là hợp bài toán |
| hidden 58 | 55.667 tham số | xấp xỉ DS-TCN-64 (56.281) và LSTM-67 (56.908) |

```
CNN-LSTM-58   55.667
DS-TCN-64     56.281
LSTM-67       56.908       CNN-LSTM nhỏ hơn 2,2%
```

Đặt cạnh **LSTM-67** để so, không phải LSTM-352.

## Giới hạn phải ghi khi báo cáo

Kiến trúc này đổi **đồng thời hai thứ**: cách trích đặc trưng, và độ dài chuỗi
đưa vào LSTM. Nếu nó thắng thì chưa biết nhờ cái nào.

Đối chứng rẻ để tách hai nguyên nhân: thay hai tầng tích chập bằng `AvgPool` rút
200 xuống 50 rồi đưa vào LSTM. Bản AvgPool cũng thắng thì công là của việc rút
ngắn chuỗi, không phải của đặc trưng tích chập. Chỉ chạy đối chứng này **nếu**
CNN-LSTM thắng.

Và như mọi cấu hình khác trong TN1, bộ tham số huấn luyện lấy từ
`optimal_params.json` của MobiVital, dò cho LSTM-352. Điều kiện này áp dụng như
nhau cho cả bốn cấu hình nhỏ nên phép so vẫn có giá trị.

## 1. Chuẩn bị Colab

Mount Drive để lấy cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó. Xem dòng `commit đồ án` để chắc đang chạy bản mới.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm bản cài đặt

`scripts/check_model.py` chạy tám phép kiểm mất vài giây. Sai phép nào là dừng
hẳn, không chạy tiếp.

Phép số 6 và 7 kiểm đúng phần dễ sai của kiến trúc này: chuỗi có rút từ 200
xuống 50 không, có đúng hai tầng tích chập cùng `stride=2` không, và không có
pooling nào lẻn vào. Phép số 8 kiểm số tham số có xấp xỉ LSTM-67 không — lệch
quá 5% thì dừng vì không so công bằng được.

In [ ]:
!python scripts/check_model.py --model cnn_lstm --hidden 58 --compare-with lstm --compare-hidden 67

## 3. CNN-LSTM-58 — 4 fold CV, seed 0 trước

Chạy **seed 0 trước**, khoảng 20 phút. Xem kết quả rồi mới quyết có chạy tiếp
seed 1 và 2 không.

Quy tắc chốt trước khi chạy, để không phải chọn ngưỡng cho vừa kết quả:

```
cv_score so với LSTM-67 (0.7532)

  cao hơn, hoặc thấp hơn dưới 0.01   ->  chạy tiếp seed 1 và 2
  thấp hơn quá 0.01                  ->  dừng, ghi vào phần hạn chế là đã thử
```

Ngưỡng 0.01 là quy tắc tiết kiệm máy, không phải kiểm định thống kê — dao động
hạt giống đo được ở bốn cấu hình trước là 0.004, nhưng kiến trúc mới có thể
nhiễu hơn, nên **có nguy cơ loại nhầm**.

Cấu hình huấn luyện giữ y nguyên: 20 epoch, Adam lr 1e-4, batch 64, MSE,
`corr` 0.9, bốn fold cũ.

In [ ]:
!python scripts/run_cv.py --experiment tn1 --model cnn_lstm --hidden 58 --seed 0

Seed 0 đạt ngưỡng thì chạy ô này. Khoảng 40 phút.

In [ ]:
!python scripts/run_cv.py --experiment tn1 --model cnn_lstm --hidden 58 --seed 1
!python scripts/run_cv.py --experiment tn1 --model cnn_lstm --hidden 58 --seed 2

## 4. Cất kết quả

`run_cv.py` đã tự nén sau mỗi fold và chép sang Drive. Ô dưới nén lại một lần
sau khi xong, ra tên riêng.

In [ ]:
!python scripts/save_results.py tn1 --out tn1_cnn_lstm_h58

## 5. Ngắt phiên

Colab giữ runtime sau khi ô cuối chạy xong và vẫn tính giờ. Kết quả đã nén sang
Drive ở mục 4 nên ngắt ở đây không mất gì.

In [ ]:
from google.colab import runtime
runtime.unassign()